# Introdução

Digamos que eu esteja me sentindo de uma maneira muito específica, e eu queira ouvir uma música que consoe meu estado de espírito. Para isso, podemos usar técnicas de Processamento de Linguagem Natural (PLN) para analisar o sentimento de uma frase ou texto e, em seguida, recomendar uma música que corresponda a esse sentimento.

Lógicamente, o universo musical é vasto e diverso, há todavia um artista com tamanho alcance e talento que faz com que todos os outros sejam irrelevantes: David Bowie.

Dessa forma, podemos criar um chatbot que, dada uma descrição do seu estado de espírito, responda a música ideal para enriquecer o seu ser.

# Dados

## Obtenção

Em primeiro lugar é necessário obter todas as letras das músicas de David Bowie. Para tanto, foi utilizado o site [Bowie Wonderworld](https://www.bowiewonderworld.com/songs/dblyrics.htm), que contém todas as letras das músicas do artista. A partir desse site, é possível realizar uma cópia bruta (é necessário desativar os scripts do site para tanto) de todas as letras para um arquivo de texto.

## Transformação e anotação

Primeiramente é necessário transformar o arquivo bruto em um arquivo estruturado, de forma que cada música seja representada por uma linha, contendo o título da música, a letra e o sentimento.

Devido ao altíssimo volume de músicas, a anotação manual de sentimentos para cada música seria inviável.
Por tanto, conclui-se que a a utilização de assistência de IA seria a solução mais apropriada.

A transformação e anotação ocorreram por tanto através do Copilot,
 utilizado o modelo GPT-5.6 Sol, janela de contexto de e raciocínio padrão com o seguinte prompt:


```
I need you to create a csv with the title of each song, the lyrics of each song, and a sentence describing the feelings of the song
```


## Aprimoração

O resultado das anotações iniciais todavia não era satisfatório, sentimentos genéricos e muitas vezes repetitivos foram atribuídos a músicas com sentimentos distintos.

As estratégias adotadas para aprimorar as anotações foram as seguintes:

1. Configurar a janela de contexto para 1.1M tokens
2. Aumentar o nível de raciocínio para "xhigh"
3. Descrever melhor o prompt, incluindo exemplos de sentimentos para músicas específicas. O prompt final utilizado foi o seguinte:

```
I need the feeling field to be more descriptive and individual to each song, for instance, Word on a Wing is a song that evokes a feeling of resignation to a higher power, a plea with God for direction, you can look the net for explanations too
```


# Pré-processamento

## Inicializando o ambiente

In [ ]:
import pandas as pd
import numpy as np

## Importando dados

In [2]:
try:
    lyrics = pd.read_csv("bowie_songs.csv")
except FileNotFoundError:
    lyrics = pd.read_csv("https://raw.githubusercontent.com/fabio-osti/maua/refs/heads/master/artificial%20intelligence/pln/atividades/bowie_songs.csv")

lyrics

,title,lyrics,feelings
0,1917,(Instrumental),"Wordless and murky, this piece leans on its Fi..."
1,1984,"Someday they won't let you, now you must agree...","A funk-driven warning shot, it wraps claustrop..."
2,1984/Dodo,"Someday they won't let you, but now you must a...","Stitching dystopian alarm to hushed gossip, th..."
3,5:15 The Angels Have Gone,"5:15\nI'm changing trains, this little town\nL...","Set on a rainy platform, this is a farewell we..."
4,'87 And Cry,It's just a one dollar secret\nA lover's secre...,"Frustration curdles into bitterness, a snarlin..."
...,...,...,...
592,Word On A Wing,"In this age of grand delusion, you walked into...","Performed live, the hymn sounds even more expo..."
593,Yassassin,CHORUS\n Yassassin - I'm not a moody guy\n Y...,"A migrant's weary plea for peace, pride and ex..."
594,You Better Tell Her,NaN,"Impatient counsel drives the phrase, somebody ..."
595,You Can't Sit Down,Hey pretty baby! (you can't sit down)\nA don't...,"Irresistible compulsion to move, the beat trea..."


## Limpando dados

In [3]:
lyrics_clean = lyrics.dropna()
lyrics_clean

,title,lyrics,feelings
0,1917,(Instrumental),"Wordless and murky, this piece leans on its Fi..."
1,1984,"Someday they won't let you, now you must agree...","A funk-driven warning shot, it wraps claustrop..."
2,1984/Dodo,"Someday they won't let you, but now you must a...","Stitching dystopian alarm to hushed gossip, th..."
3,5:15 The Angels Have Gone,"5:15\nI'm changing trains, this little town\nL...","Set on a rainy platform, this is a farewell we..."
4,'87 And Cry,It's just a one dollar secret\nA lover's secre...,"Frustration curdles into bitterness, a snarlin..."
...,...,...,...
591,Without You I'm Nothing,Strange infatuation seems to grace the evening...,"Sultry self-abasement, decadent images sliding..."
592,Word On A Wing,"In this age of grand delusion, you walked into...","Performed live, the hymn sounds even more expo..."
593,Yassassin,CHORUS\n Yassassin - I'm not a moody guy\n Y...,"A migrant's weary plea for peace, pride and ex..."
595,You Can't Sit Down,Hey pretty baby! (you can't sit down)\nA don't...,"Irresistible compulsion to move, the beat trea..."


# Testes

Embora a tarefa seja subjetiva, é necessário, ainda assim, ao menos um teste que demonstre o comportamento do modelo.

Para tanto, foram criadas algumas sentenças descrevendo sentimentos específicos, e a música esperada para cada sentimento.

In [117]:
_test = [
    # Praticamente uma cópia do sentimento da música, serve como teste de sanidade
    ("I am on a disordered pilgrimage from occult dread toward desperate romance, a numbed figure pushing himself to feel anything at all.",
     "Station To Station"),
    # Esse pode ser acertado tanto pela letra quanto pelo sentimento
    ("I feel like I'm cracking under pressure and that only love can save me.", "Under Pressure"),
    # Relativamente fácil também, mas dá para errar
    ("I feel euphoric and want to dance with my beloved.", "Let's Dance"),
    # Começa a dificultar, as palavras não estão diretamente nem na letra nem no sentimento
    ("I'm feeling regretful and ashamed for falling so low on my addiction.", "Ashes To Ashes"),
    ("I feel resignation, I just want to understand God's plan for me.", "Word On A Wing"),
    # Esse é o mais difícil, é necessário interpretar o contexto
    ("I'm completely head over heels in love and want to deliver myself completely", "I Would Be Your Slave"),
    # Extremamente específica
    ("I'm in love with a chinese woman", "China Girl"),
    # Bonus, não tem resposta certa, mas é interessante ver o que o modelo sugere
    ("I'm feeling anxious and stressed about an upcoming event.", "???"),
]

def _ranking(indices: tuple[list[int], list[float]]):
    return '\n'.join(
        f"\t\t{i}. {lyrics_clean.iloc[song_idx]['title']}: {sim:.4f}"
        for i, (song_idx, sim) in enumerate(zip(indices[0][:3], indices[1][:3]), start=1)
    )

def _position(expected, feeling, lyrics):
    if expected not in lyrics_clean['title'].values:
        return -1, -1
    i = np.where(lyrics_clean['title'] == expected)[0][0]
    return (
        np.where(feeling[0] == i)[0][0] + 1,
        np.where(lyrics[0] == i)[0][0] + 1,
    )

def test_similarity_function(feeling_similarity_function, lyrics_similarity_function):
    f_rank = 0
    l_rank = 0
    c = 0
    for phrase, expected in _test:
        most_similar_by_feeling = feeling_similarity_function(phrase)
        most_similar_by_lyrics = lyrics_similarity_function(phrase)
        print(f"Input phrase: {phrase}")
        if expected != "???":
            (feeling_pos, lyrics_pos) = _position(expected, most_similar_by_feeling, most_similar_by_lyrics)
            print(f"\t* Expected song ({expected}) true position:\n\t\t* By feeling: {feeling_pos}º\n\t\t* By lyrics: {lyrics_pos}º")
            f_rank += feeling_pos
            l_rank += lyrics_pos
            c += 1
        print(f"\t* Most similar songs by feeling: \n{_ranking(most_similar_by_feeling)}")
        print(f"\t* Most similar songs by lyrics: \n{_ranking(most_similar_by_lyrics)}")
        print("\n")
    print(f"Average rank by feeling: {f_rank / c}")
    print(f"Average rank by lyrics: {l_rank / c}")

# Rankeando por similaridade

A abordagem principal do projeto é simples, comparar a similaridade da sentença descrevendo o sentimento do usuário com a sentença descrevendo o sentimento de cada música e também com a própria letra da música.

Para tanto, é necessário vetorizar as sentenças, utilizaremos diferentes técnicas de vetorização para comparar os resultados e escolher a melhor abordagem. A comparação será feita utilizando a métrica de similaridade cosseno, que mede o ângulo entre dois vetores, sendo 1 para vetores idênticos e 0 para vetores ortogonais.

In [52]:
from sklearn.metrics.pairwise import cosine_similarity

def similaridade(de, para):
    similaridades = cosine_similarity(de, para)
    argsoted = similaridades.argsort()[0][::-1]
    return similaridades.argsort()[0][::-1], similaridades[0][argsoted]

# Vetorização

Para realizar a vetorização das sentenças, utilizaremos diferentes técnicas de vetorização, como TF-IDF, Word2Vec, Doc2Vec e Transformers. Cada técnica possui suas próprias características e vantagens, e será interessante comparar os resultados obtidos com cada uma delas.

## TF-IDF

A primeira abordagem, que servirá de base de comparação para as demais, é a utilização de TF-IDF (Term Frequency-Inverse Document Frequency) para transformar os sentimentos das músicas. Essa técnica permite identificar a importância de cada palavra em relação ao conjunto de documentos (neste caso, os sentimentos das músicas).

### Treinando o modelo

Começamos treinando o modelo TF-IDF para os sentimentos e para as letras das músicas, utilizando a biblioteca `sklearn`. A função `TfidfVectorizer` é utilizada para transformar o texto em uma matriz de TF-IDF.

Como o modelo TF-IDF já da um peso menor a palavras comuns, não é necessário remover stopwords, mas é possível fazer isso caso seja desejado.

Além disso, o modelo TF-IDF detém um próprio tokenizador, dessa forma, não é necessário implementar um tokenizador para essa abordagem.

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

feelings_tfidf_vectorizer = TfidfVectorizer()
feelings_tfidf_matrix = feelings_tfidf_vectorizer.fit_transform(lyrics_clean['feelings'])

lyrics_tfidf_vectorizer = TfidfVectorizer()
lyrics_tfidf_matrix = lyrics_tfidf_vectorizer.fit_transform(lyrics_clean['lyrics'])

### Implementando a função de similaridade

Vetorizamos a sentença de entrada utilizando o mesmo vetor TF-IDF utilizado para os sentimentos das músicas e calculamos a similaridade cosseno entre a sentença vetorizada e a matriz de sentimentos das músicas. A função `most_similar_tfidf` retorna os índices das músicas mais similares à sentença de entrada.

In [47]:
def most_similar_tfidf(sentence, matrix, vectorizer):
    phrase_vec = vectorizer.transform([sentence])
    return similaridade(phrase_vec, matrix)

def most_similar_by_feeling_tfidf(sentence):
    return most_similar_tfidf(sentence, feelings_tfidf_matrix, feelings_tfidf_vectorizer)

def most_similar_by_lyrics_tfidf(sentence):
    return most_similar_tfidf(sentence, lyrics_tfidf_matrix, lyrics_tfidf_vectorizer)

### Testando a função de similaridade

In [118]:
test_similarity_function(
    most_similar_by_feeling_tfidf,
    most_similar_by_lyrics_tfidf,
)

Input phrase: I am on a disordered pilgrimage from occult dread toward desperate romance, a numbed figure pushing himself to feel anything at all.
	* Expected song (Station To Station) true position:
		* By feeling: 1º
		* By lyrics: 180º
	* Most similar songs by feeling: 
		1. Station To Station: 0.9618
		2. Goodbye Mr. Ed: 0.1374
		3. Soul Love: 0.1313
	* Most similar songs by lyrics: 
		1. I Am With Name: 0.1452
		2. The Pretty Things Are Going To Hell: 0.1178
		3. When The Wind Blows: 0.1035


Input phrase: I feel like I'm cracking under pressure and that only love can save me.
	* Expected song (Under Pressure) true position:
		* By feeling: 72º
		* By lyrics: 1º
	* Most similar songs by feeling: 
		1. Cat People (Putting Out Fire): 0.3557
		2. "Helden": 0.2442
		3. Girls: 0.1927
	* Most similar songs by lyrics: 
		1. Under Pressure: 0.2238
		2. Under The God: 0.1800
		3. Love Me Do: 0.1458


Input phrase: I feel euphoric and want to dance with my beloved.
	* Expected song (Let's D

Podemos observar que a função de similaridade baseada em TF-IDF apresenta resultados razoáveis quando as palavras da sentença de entrada estão presentes na letra ou no sentimento da música. Todavia, pequenas variações acabam deteriorando a performance do modelo, o que pode ser muito bem verificado no teste de China Girl, pois, da sentença do usuário, as palavras "chinese" e "woman" não estão presentes na letra da música, o que faz com que o modelo não consiga identificar a música correta.

## Embedding

Fazendo o uso de embeddings pré-treinados, podemos melhorar a performance do modelo, pois embeddings são representações vetoriais de palavras ou frases que capturam o significado semântico das palavras, permitindo que palavras com significados semelhantes tenham representações vetoriais próximas no espaço vetorial. Dessa forma, mesmo que as palavras da sentença de entrada não estejam presentes na letra ou no sentimento da música, o modelo ainda pode identificar músicas com sentimentos semelhantes.

### Tokenização

Como os métodos de embedding Word2Vec e Doc2Vec não contem tokenizador próprio, é necessário implementar um tokenizador.

In [113]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

def tokenize(text, remove_stopwords=True):
    tokens = word_tokenize(text.lower())
    if remove_stopwords:
        tokens = [t for t in tokens if t.isalpha() and t not in stopwords.words('english')]
    else:
        tokens = [t for t in tokens if t.isalpha()]
    return tokens

[nltk_data] Downloading package punkt to /home/fabio/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /home/fabio/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


### Word2Vec

Para o método Word2Vec, utilizaremos o modelo pré-treinado do Google News, que contém vetores de palavras treinados em um grande corpus de notícias. Esse modelo é capaz de capturar relações semânticas entre palavras, permitindo que palavras com significados semelhantes tenham representações vetoriais próximas no espaço vetorial.

In [114]:
import gensim.downloader as api

# Load pre-trained Word2Vec model
word2vec_model = api.load("word2vec-google-news-300")
word2vec_model["love"][::10] # Example of getting the vector for a word

array([ 0.10302734, -0.02624512, -0.03857422, -0.11181641,  0.18652344,
       -0.10986328,  0.24316406,  0.28515625, -0.31054688, -0.00958252,
       -0.19726562, -0.22167969,  0.05566406, -0.140625  , -0.09375   ,
        0.27734375, -0.07763672,  0.06005859, -0.30664062,  0.10644531,
       -0.0390625 , -0.10839844, -0.07128906, -0.24804688,  0.04736328,
        0.07470703, -0.09179688,  0.07763672,  0.16113281, -0.03198242],
      dtype=float32)

#### Vetorização

A vetorização de sentença é feita calculando a média dos vetores das palavras que compõem a sentença. Se uma palavra não estiver presente no vocabulário do modelo, ela é ignorada. Se nenhuma palavra da sentença estiver presente no vocabulário, um vetor de zeros é retornado.

Aqui, removeremos as stopwords, pois elas não contribuem para o significado da sentença e podem distorcer a média dos vetores das palavras.

In [ ]:
from gensim.models import KeyedVectors

def avg_w2v_vec(text, w2v_model: KeyedVectors):
    tokens = tokenize(text.lower())
    vectors = [
        w2v_model[t]
        for t in tokens
        if t in w2v_model
    ]

    if not vectors:
        return np.zeros(w2v_model.vector_size)

    return np.mean(vectors, axis=0)

#### Implementando a função de similaridade

A função de similaridade é implementada da mesma forma que a função de similaridade baseada em TF-IDF, mas utilizando a vetorização de sentença baseada em Word2Vec.

Como a etapa de transform não foi realizada no modelo, é necessário vetorizar cada sentença para criar uma matriz de vetores de sentimentos e uma matriz de vetores de letras.

In [115]:
feelings_word2vec_matrix = [
    avg_w2v_vec(feeling, word2vec_model)
    for feeling in lyrics_clean['feelings']
]

lyrics_word2vec_matrix = [
    avg_w2v_vec(lyric, word2vec_model)
    for lyric in lyrics_clean['lyrics']
]

def most_similar_w2v(sentence, matrix):
    phrase_vec = avg_w2v_vec(sentence, word2vec_model).reshape(1, -1)
    return similaridade(phrase_vec, matrix)

def most_similar_by_feeling_w2v(sentence):
    return most_similar_w2v(sentence, feelings_word2vec_matrix)

def most_similar_by_lyrics_w2v(sentence):
    return most_similar_w2v(sentence, lyrics_word2vec_matrix)

In [116]:
test_similarity_function(
    most_similar_by_feeling_w2v,
    most_similar_by_lyrics_w2v
)

Input phrase: I am on a disordered pilgrimage from occult dread toward desperate romance, a numbed figure pushing himself to feel anything at all.
	* Expected song (Station To Station) true position:
		* By feeling: 1º
		* By lyrics: 44º
	* Most similar songs by feeling: 
		1. Station To Station: 0.9928
		2. Aladdin Sane (1913-1938-197?): 0.6942
		3. 1917: 0.6942
	* Most similar songs by lyrics: 
		1. The Supermen: 0.7194
		2. Too Dizzy: 0.6927
		3. That's Motivation: 0.6924


Input phrase: I feel like I'm cracking under pressure and that only love can save me.
	* Expected song (Under Pressure) true position:
		* By feeling: 6º
		* By lyrics: 303º
	* Most similar songs by feeling: 
		1. Knock On Wood: 0.6868
		2. Don't Look Down: 0.6613
		3. Bars Of The County Jail - (demo): 0.6527
	* Most similar songs by lyrics: 
		1. Letter To Hermione: 0.7547
		2. I'm Not Quite - (demo): 0.7536
		3. Sweet Thing - (working lyrics): 0.7349


Input phrase: I feel euphoric and want to dance with my bel

### Word2Vec + TF-IDF

Utilizar o Word2Vec sozinho pode não ser suficiente para capturar a importância relativa das palavras na sentença. Para melhorar a vetorização, podemos combinar o Word2Vec com o TF-IDF, ponderando os vetores das palavras pelo seu peso TF-IDF. Dessa forma, palavras mais importantes na sentença terão maior influência no vetor final.

Como utilizaremos o TF-IDF para ponderar os vetores das palavras, não é necessário remover as stopwords, pois o TF-IDF já atribui um peso menor a palavras comuns.

#### Vetorizador

In [ ]:
def weighted_avg_w2v_vec(sentence, w2v_model, tfidf):
    words = tokenize(sentence.lower(), False)
    vectors = []
    weights = []

    for word in words:
        if word in w2v_model and word in tfidf:
            vectors.append(w2v_model[word])
            weights.append(tfidf[word])

    if not vectors:
        return np.zeros(w2v_model.vector_size)

    return np.average(vectors, axis=0, weights=weights)

#### Função de similaridade

In [79]:
feeling_weighted_word2vec_matrix = [
    weighted_avg_w2v_vec(feeling, word2vec_model, feelings_tfidf_vectorizer.vocabulary_)
    for feeling in lyrics_clean['feelings']
]

lyrics_weighted_word2vec_matrix = [
    weighted_avg_w2v_vec(lyric, word2vec_model, lyrics_tfidf_vectorizer.vocabulary_)
    for lyric in lyrics_clean['lyrics']
]


def most_similar_weighted_w2v(sentence, matrix, vocabulary):
    phrase_vec = weighted_avg_w2v_vec(sentence, word2vec_model, vocabulary).reshape(1, -1)
    similarities = cosine_similarity(phrase_vec, matrix)
    return similarities.argsort()[0][::-1]


def most_similar_by_feeling_weighted_w2v(sentence):
    return most_similar_weighted_w2v(
        sentence, feeling_weighted_word2vec_matrix, feelings_tfidf_vectorizer.vocabulary_
    )


def most_similar_by_lyrics_weighted_w2v(sentence):
    return most_similar_weighted_w2v(
        sentence, lyrics_weighted_word2vec_matrix, lyrics_tfidf_vectorizer.vocabulary_
    )

NameError: name 'weighted_avg_w2v_vec' is not defined

In [ ]:
test_similarity_function(
    most_similar_by_feeling_weighted_w2v,
    most_similar_by_lyrics_weighted_w2v
)

### Doc2Vec

In [88]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

feelings_tags = [
    TaggedDocument(words=word_tokenize(feeling.lower()), tags=[str(i)])
    for i, feeling in enumerate(lyrics_clean['feelings'])
]

feelings_doc2vec = Doc2Vec(vector_size=100, window=5, min_count=1, workers=4, epochs=40)
feelings_doc2vec.build_vocab(feelings_tags)
feelings_doc2vec.train(feelings_tags, total_examples=feelings_doc2vec.corpus_count, epochs=feelings_doc2vec.epochs)

lyrics_tags = [
    TaggedDocument(words=word_tokenize(lyric.lower()), tags=[str(i)])
    for i, lyric in enumerate(lyrics_clean['lyrics'])
]

lyrics_doc2vec = Doc2Vec(vector_size=100, window=5, min_count=1, workers=4, epochs=40)
lyrics_doc2vec.build_vocab(lyrics_tags)
lyrics_doc2vec.train(lyrics_tags, total_examples=lyrics_doc2vec.corpus_count, epochs=lyrics_doc2vec.epochs)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [110]:
def most_similar_doc2vec(sentence, matrix):
    inferred_vector = matrix.infer_vector(word_tokenize(sentence.lower()))
    most_similar = matrix.dv.most_similar([inferred_vector], topn=len(matrix.dv))
    idxs = []
    sims = []
    for tag, sim in most_similar:
        idxs.append(int(tag))
        sims.append(sim)
    return idxs, sims

def most_similar_by_feeling_doc2vec(sentence):
    return most_similar_doc2vec(sentence, feelings_doc2vec)

def most_similar_by_lyrics_doc2vec(sentence):
    return most_similar_doc2vec(sentence, lyrics_doc2vec)

In [112]:
test_similarity_function(
    most_similar_by_feeling_doc2vec,
    most_similar_by_lyrics_doc2vec
)

Input phrase: I am on a disordered pilgrimage from occult dread toward desperate romance, a numbed figure pushing himself to feel anything at all.
	* Expected song (Station To Station) true position:
		* By feeling: 1º
		* By lyrics: 288º
	* Most similar songs by feeling: 
		1. Station To Station: 0.9947
		2. Time Will Crawl: 0.9936
		3. Ashes To Ashes: 0.9935
	* Most similar songs by lyrics: 
		1. Crystal Japan: 0.5691
		2. Brussels: 0.5416
		3. Golden Years (Instrumental): 0.5293


Input phrase: I feel like I'm cracking under pressure and that only love can save me.
	* Expected song (Under Pressure) true position:
		* By feeling: 89º
		* By lyrics: 9º
	* Most similar songs by feeling: 
		1. Survive: 0.8594
		2. Within You: 0.8513
		3. Modern Love: 0.8369
	* Most similar songs by lyrics: 
		1. Soul Love - (demo): 0.4428
		2. The Wedding: 0.4224
		3. What Kind Of Fool Am I?: 0.4189


Input phrase: I feel euphoric and want to dance with my beloved.
	* Expected song (Let's Dance) true po

In [98]:
def most_cos_similar_doc2vec(sentence, model):
    inferred_vector = model.infer_vector(word_tokenize(sentence.lower()))
    return similaridade(inferred_vector.reshape(1, -1), model.dv.vectors)

def most_cos_similar_by_feeling_doc2vec(sentence):
    return most_cos_similar_doc2vec(sentence, feelings_doc2vec)

def most_cos_similar_by_lyrics_doc2vec(sentence):
    return most_cos_similar_doc2vec(sentence, lyrics_doc2vec)

In [99]:
test_similarity_function(
    most_cos_similar_by_feeling_doc2vec,
    most_cos_similar_by_lyrics_doc2vec
)

Input phrase: I am on a disordered pilgrimage from occult dread toward desperate romance, a numbed figure pushing himself to feel anything at all.
	* Expected song (Station To Station) true position:
		* By feeling: 1º
		* By lyrics: 216º
	* Most similar songs by feeling: 
		1. Station To Station: 0.9959
		2. Ashes To Ashes: 0.9954
		3. O Superman: 0.9952
	* Most similar songs by lyrics: 
		1. Crystal Japan: 0.5764
		2. Brussels: 0.5584
		3. The Wedding: 0.5487


Input phrase: I feel like I'm cracking under pressure and that only love can save me.
	* Expected song (Under Pressure) true position:
		* By feeling: 86º
		* By lyrics: 3º
	* Most similar songs by feeling: 
		1. Survive: 0.8886
		2. Within You: 0.8816
		3. Modern Love: 0.8674
	* Most similar songs by lyrics: 
		1. Soul Love - (demo): 0.4616
		2. The Wedding: 0.4296
		3. Under Pressure: 0.4284


Input phrase: I feel euphoric and want to dance with my beloved.
	* Expected song (Let's Dance) true position:
		* By feeling: 44º
		

## Transformers

In [30]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("google/embeddinggemma-300m")

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 1630.06it/s]


In [31]:
feelings_transformer_matrix = model.encode(lyrics_clean['feelings'].to_list())

lyrics_transformer_matrix = model.encode(lyrics_clean['lyrics'].to_list())

In [39]:
def most_similar_transformer(sentence, matrix):
    phrase_vec = model.encode([sentence])
    return similaridade(phrase_vec, matrix)

def most_similar_by_feeling_transformer(sentence):
    return most_similar_transformer(sentence, feelings_transformer_matrix)

def most_similar_by_lyrics_transformer(sentence):
    return most_similar_transformer(sentence, lyrics_transformer_matrix)


In [43]:
test_similarity_function(
    most_similar_by_feeling_transformer,
    most_similar_by_lyrics_transformer
)

KeyboardInterrupt: 